# 🔍 GROOPY — Exploring the ASL Alphabet
### CRISP-DM: Data Understanding

Before training anything, we look at the data: **how many images per class**, **what the signs look like**, **image properties**, and the **known hard-to-tell-apart letters**. These findings drive the preprocessing + augmentation choices used in the bake-off (`02_bakeoff.ipynb`).

> Run this **before** the bake-off. It's quick (no training) and the plots are great evidence of the Data-Understanding step for your report.

## 1 · Setup
Clone the repo, install deps, and download the dataset. Needs a Kaggle key (kaggle.com/settings → **Create Legacy API Key** → `kaggle.json`), and accept the [ASL Alphabet](https://www.kaggle.com/datasets/grassknoted/asl-alphabet) terms once.

In [ ]:
!git clone https://github.com/SpliiiiT/GROOPY.git 2>/dev/null || (cd GROOPY && git pull -q)
%cd /content/GROOPY
import sys; sys.path.insert(0, '/content/GROOPY')
!pip install -q kaggle

# upload kaggle.json, then download (skip if the data is already present)
from google.colab import files; files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!python data/download_asl_alphabet.py

## 2 · How many images per class?
A balanced dataset means accuracy isn't skewed toward over-represented classes. ASL Alphabet is usually ~3000 per class.

In [ ]:
import os, collections
import matplotlib.pyplot as plt
from recognition.src.config import ASL_TRAIN_DIR

counts = {cls: len(os.listdir(ASL_TRAIN_DIR / cls)) for cls in sorted(os.listdir(ASL_TRAIN_DIR))}
print(f'{len(counts)} classes, {sum(counts.values()):,} images total, '
      f'{min(counts.values())}-{max(counts.values())} per class')

plt.figure(figsize=(12, 4))
plt.bar(counts.keys(), counts.values())
plt.xticks(rotation=90); plt.ylabel('images'); plt.title('Images per class')
plt.tight_layout(); plt.show()

## 3 · What do the signs look like?
One example per class — a sanity check that labels match the handshapes, and a feel for lighting/background variation.

In [ ]:
import cv2
classes = sorted(counts)
fig, axes = plt.subplots(5, 6, figsize=(14, 12))
for ax, cls in zip(axes.ravel(), classes):
    p = next((ASL_TRAIN_DIR / cls).glob('*'))
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax.set_title(cls); ax.axis('off')
for ax in axes.ravel()[len(classes):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

## 4 · Image properties
Confirm the images are a consistent size/format so the pipeline can resize them uniformly to 224×224.

In [ ]:
import numpy as np
sample = [cv2.imread(str(next((ASL_TRAIN_DIR / c).glob('*')))) for c in classes[:10]]
shapes = {s.shape for s in sample}
print('sample image shapes (H, W, C):', shapes)
print('dtype:', sample[0].dtype, '| pixel range:', int(sample[0].min()), '-', int(sample[0].max()))

## 5 · Takeaways → Data Preparation
These observations shape the modeling pipeline:

- **Balanced classes** → plain accuracy is a fair metric (we still report macro-F1).
- **Consistent image size** → resize everything to **224×224** (what the pretrained backbones expect).
- **No horizontal flip** in augmentation — mirroring changes some handshapes into different/invalid letters.
- **Hard-to-distinguish pairs** (e.g. **M/N/S/T**, **A/E**) look alike — expect confusion there; we check the confusion matrix + Grad-CAM in `02_bakeoff.ipynb`.
- **Background/lighting varies** → crop to the hand before classifying (see `preprocess.py`) so the model learns the *handshape*, not the room.

Next: **`02_bakeoff.ipynb`** — train and compare the models.